# MAIB BERT pre-training on Kaggle

Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On**, and under *Input* the dataset with
`maib-work.zip` (first session) or the output of the previous version of THIS notebook (later sessions).

Then **Save Version -> Save & Run All (Commit)**: it runs in the background for up to 12 h, the browser can be
closed. Everything the run produces is in the output of that version. See cloud/README.md.

In [ ]:
# 1) Copy the project to the writable folder /kaggle/working.
#    The inputs are read-only. If several copies are attached (the uploaded dataset and the output of an earlier
#    session), take the one that got furthest: most finished models, results and checkpoints.
import glob, os, shutil

WORK = "/kaggle/working/maib-bert-pretraining"

def progress(folder):
    return (100 * len(glob.glob(folder + "/models/*/config.json")) + 10 * len(glob.glob(folder + "/results/*.json"))
            + len(glob.glob(folder + "/**/*.resume.pt", recursive=True)))

if not os.path.exists(WORK):
    copies = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/maib-bert-pretraining/run_pipeline.py", recursive=True)]
    if not copies:                       # Kaggle normally unpacks an uploaded zip; if it did not, unpack it here
        for z in glob.glob("/kaggle/input/**/maib-work.zip", recursive=True):
            shutil.unpack_archive(z, "/kaggle/tmp_unzip")
        copies = [os.path.dirname(p) for p in glob.glob("/kaggle/tmp_unzip/**/maib-bert-pretraining/run_pipeline.py", recursive=True)]
    if not copies:
        raise SystemExit("The project is not among the inputs: add the dataset (maib-work.zip) or the previous output.")
    for c in copies:
        print(f"found {c}  (progress {progress(c)})")
    source = max(copies, key=progress)
    print("copying", source)
    shutil.copytree(source, WORK)
os.chdir(WORK)
print(sorted(os.listdir("results")) if os.path.exists("results") else "no results yet")

In [ ]:
# 2) Check the GPU (must show a Tesla T4) and the libraries.
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: set Accelerator = GPU T4 x2 in the settings."

In [ ]:
# 3) Run (or continue) the experiment for at most 10.5 h, then stop cleanly, so there is time to save the output
#    before Kaggle's 12-h limit. Finished work is kept; an unfinished run continues next session after its last
#    finished epoch. -s INT = like Ctrl+C: the running step is stopped too, never left half-written.
#    TQDM_DISABLE=1: no progress bars (they fill the log); every epoch still prints its result.
!TQDM_DISABLE=1 timeout -s INT 37800 python run_pipeline.py 2>&1 | tee -a run_full.log

In [ ]:
# 4) Where are we? (results so far; the files stay in the output of this version)
!tail -n 40 reports/comparison.md 2>/dev/null || echo "no comparison yet"
!ls models results
!du -sh /kaggle/working